# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: Random Forest classifier**, compared against Logistic Regression as a simpler
reference point.

**Why it fits this lane:** the question is binary classification (declining now vs not),
and I already have direct evidence from the starter pipeline that tree-based models beat
both a hand rule and a linear model on this exact task (Precision@50: 0.240 baseline →
0.740 random forest). A random forest can pick up interactions between signals (e.g.
position + CTR + consistency together) that a hand-written rule or a linear model can't
easily capture, without needing heavy tuning.

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from huggingface_hub import login, notebook_login

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN")

data_files = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
dataset = load_dataset("parquet", data_files=data_files, token=HF_TOKEN)
df_slice = dataset["train"].to_pandas()
df_slice["report_date"] = pd.to_datetime(df_slice["report_date"])

# Aggregate to one row per page
page = df_slice.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_with_data=("report_date", "nunique"),
).reset_index()
page["ctr"] = np.where(page["impressions"] > 0, page["clicks"] / page["impressions"], 0)

# Label: first-half vs second-half clicks
median_date = df_slice["report_date"].median()
first_half = df_slice[df_slice["report_date"] < median_date]
second_half = df_slice[df_slice["report_date"] >= median_date]
first_clicks = first_half.groupby("content_hash_id")["gsc_clicks"].sum()
second_clicks = second_half.groupby("content_hash_id")["gsc_clicks"].sum()
page["first_half_clicks"] = page["content_hash_id"].map(first_clicks).fillna(0)
page["second_half_clicks"] = page["content_hash_id"].map(second_clicks).fillna(0)
page["is_declining_label"] = (page["second_half_clicks"] < page["first_half_clicks"]).astype(int)

print(f"Pages: {len(page):,} | Declining rate: {page['is_declining_label'].mean():.1%}")
print("Method: Random Forest (vs Logistic Regression reference)")

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Pages: 331,437 | Declining rate: 8.7%
Method: Random Forest (vs Logistic Regression reference)


## 2. Split design

**Grouped by client** — I hold out ~20% of `client_hash_id`s entirely, so no client's pages
appear in both train and test.

**Why this is honest for my question:** pages from the same client tend to share patterns
(same CMS, same content strategy, same seasonality). A random row-level split would let the
model partly "memorize" a client's style from the train set and get an inflated score on
that same client's held-out pages. A client-holdout split is a fairer test of whether the
model generalizes to clients it has never seen — which is what actually matters in
production, since a real refresh queue runs on new clients too.

In [ ]:
RANDOM_STATE = 42
feature_cols = ["impressions", "clicks", "avg_position", "ctr", "days_with_data"]

X = page[feature_cols].fillna(0)
y = page["is_declining_label"]

clients = page["client_hash_id"].unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)
test_clients = set(shuffled[: max(1, int(len(shuffled) * 0.2))])
test_mask = page["client_hash_id"].isin(test_clients)

X_train, X_test = X[~test_mask], X[test_mask]
y_train, y_test = y[~test_mask], y[test_mask]

print(f"Train clients: {len(clients) - len(test_clients)} | Test clients: {len(test_clients)}")
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Train declining rate: {y_train.mean():.1%} | Test declining rate: {y_test.mean():.1%}")

Train clients: 44 | Test clients: 11
Train rows: 269,799 | Test rows: 61,638
Train declining rate: 7.7% | Test declining rate: 13.5%


## 3. Train + compare vs my baseline

Same data, same metric (Precision@50, ROC-AUC), same client-holdout split as my Week-4
baseline (ML-07).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, f1_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return float(np.array(y_true)[order].mean()) if len(order) else 0.0

# Baseline score (same formula as ML-07), computed on the full page set then sliced to test
demand = page["impressions"].rank(pct=True)
low_consistency = 1 - page["days_with_data"].rank(pct=True)
has_position = page["avg_position"] > 0
position_opportunity = pd.Series(0.0, index=page.index)
position_opportunity[has_position] = (1 - page.loc[has_position, "avg_position"].clip(upper=50) / 50).rank(pct=True)
baseline_score = (0.45 * demand + 0.35 * low_consistency + 0.20 * position_opportunity) * 100
baseline_test_scores = baseline_score[test_mask].to_numpy()

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

results = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    results.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, probs),
        "precision_at_20": precision_at_k(y_test, probs, 20),
        "precision_at_50": precision_at_k(y_test, probs, 50),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
    })

results.append({
    "model": "baseline_rule",
    "roc_auc": roc_auc_score(y_test, baseline_test_scores),
    "precision_at_20": precision_at_k(y_test, baseline_test_scores, 20),
    "precision_at_50": precision_at_k(y_test, baseline_test_scores, 50),
    "recall": np.nan,
    "f1": np.nan,
})

results_df = pd.DataFrame(results).round(3)
display(results_df)

rf_p50 = results_df.loc[results_df["model"] == "random_forest", "precision_at_50"].iloc[0]
base_p50 = results_df.loc[results_df["model"] == "baseline_rule", "precision_at_50"].iloc[0]
lift = rf_p50 / base_p50 if base_p50 > 0 else float("nan")
print(f"\nRandom forest Precision@50: {rf_p50:.3f} vs baseline {base_p50:.3f} ({lift:.1f}x lift)")

,model,roc_auc,precision_at_20,precision_at_50,recall,f1
0,logistic_regression,0.885,0.3,0.36,0.811,0.544
1,random_forest,0.918,0.4,0.44,0.999,0.619
2,baseline_rule,0.818,0.0,0.02,NaN,NaN



Random forest Precision@50: 0.440 vs baseline 0.020 (22.0x lift)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric
table.*

In [ ]:
best_model = fitted_models["random_forest"]

# Feature importance
importance = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("--- FEATURE IMPORTANCE ---")
display(importance.to_frame("importance").round(3))

# Error analysis on the test set
test_page = page[test_mask].copy()
test_page["probability"] = best_model.predict_proba(X_test)[:, 1]
test_page["prediction"] = (test_page["probability"] >= 0.5).astype(int)

false_positives = test_page[(test_page["prediction"] == 1) & (test_page["is_declining_label"] == 0)]
false_negatives = test_page[(test_page["prediction"] == 0) & (test_page["is_declining_label"] == 1)]

print(f"\nFalse positives (flagged declining, actually fine): {len(false_positives):,} of {len(test_page):,} test rows")
print(f"False negatives (missed a real decline): {len(false_negatives):,} of {len(test_page):,} test rows")

print("\n--- SAMPLE FALSE NEGATIVES (the more costly error per my Week-2 framing) ---")
display(false_negatives[["content_hash_id", "impressions", "avg_position", "ctr", "days_with_data", "probability"]].head(10))

print("\nInterpretation: the model leans most heavily on",
      f"'{importance.index[0]}' and '{importance.index[1]}'.",
      "False negatives tend to be pages with moderate signals across the board rather than one "
      "extreme value — the kind a simple threshold rule would also miss, which is consistent "
      "with why the model beats the baseline overall but isn't perfect on borderline cases.")

--- FEATURE IMPORTANCE ---


,importance
ctr,0.404
clicks,0.380
impressions,0.145
avg_position,0.058
days_with_data,0.014



False positives (flagged declining, actually fine): 10,236 of 61,638 test rows
False negatives (missed a real decline): 10 of 61,638 test rows

--- SAMPLE FALSE NEGATIVES (the more costly error per my Week-2 framing) ---


,content_hash_id,impressions,avg_position,ctr,days_with_data,probability
49699,content_26a4c04b695c49d3,314,10.522875,0.006369,29,0.471494
63617,content_315ed4718f79dac3,4120,3.159753,0.009951,29,0.462406
100112,content_4d87554fc76b2f95,1036,5.864773,0.009653,29,0.470292
127462,content_62abae31043a04fe,2048,6.048037,0.010254,29,0.414042
129724,content_646db38c61051af6,1144,2.321316,0.025350,29,0.400557
153180,content_768b4643515b52d5,3479,6.347836,0.006611,29,0.498655
157269,content_79af90d2a1c02194,1814,4.488059,0.007166,29,0.460182
191172,content_93d9d462854364da,3621,4.718423,0.006628,29,0.480352
252111,content_c2ed3b73c9024928,572,1.958705,0.003497,29,0.449314
285822,content_dce358dc100e158d,509,4.166982,0.003929,29,0.486741



Interpretation: the model leans most heavily on 'ctr' and 'clicks'. False negatives tend to be pages with moderate signals across the board rather than one extreme value — the kind a simple threshold rule would also miss, which is consistent with why the model beats the baseline overall but isn't perfect on borderline cases.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.